## Planning:
- Load data
- Maximum Parameter with Optuna
- Blending with SLSQP
- Output

In [1]:
import numpy as np
import pandas as pd
import optuna
from sklearn.metrics import roc_auc_score

# import numpy as np
from scipy.optimize import minimize
# from sklearn.metrics import roc_auc_score

# import OS
import os

In [3]:
from pathlib import Path

In [4]:
current_dir = Path.cwd()
competition_root = current_dir.parents[1]

In [5]:
competition_path = competition_root/"Predicting Credit Card Fraud"

In [6]:
train = pd.read_csv(competition_path/"data/train.csv")
train_data = train[["id","is_fraud"]]

test = pd.read_csv(competition_path/"data/test.csv")
test_data = test[["id"]]

submit = pd.read_csv(competition_path/"data/sample_submission.csv")

In [7]:
folder_oof = competition_path/"oof"
folder_sub = competition_path/"sub"

In [8]:
content_oof = os.listdir(folder_oof)
content_sub = os.listdir(folder_sub)

In [10]:
for oof in content_oof:
    path_oof = folder_oof/oof
    oof_data = pd.read_csv(path_oof)
    train_data = pd.concat([train_data,oof_data.iloc[:,[1]]],axis=1)

In [11]:
for sub in content_sub:
    path_sub = folder_sub/sub
    sub_data = pd.read_csv(path_sub)
    test_data = pd.concat([test_data,sub_data.iloc[:,[1]]],axis=1)

## Data Process for Modeling

In [12]:
X = train_data.drop(columns=["id","is_fraud"])
y = train_data["is_fraud"]
X_test = test_data.drop(columns=["id"])
X_test.columns = X.columns

In [13]:
num_models = X.shape[1]

In [14]:
num_models

37

In [15]:
# 2. Define the objective function with a penalty constraint
def objective_slsqp(weights):
    blend = X.values @ weights
    return 1.0 - roc_auc_score(y.values, blend)

In [16]:
# 3. Setup and Run the Optimizer
# Start with equal weights for all 3 models
init_weights = np.array([1/num_models]*num_models)

In [17]:
# Define bounds (0 to 1 for each weight) and constraint (sum to 1)
bounds = [(0.0, 1.0) for _ in range(num_models)]
cons = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}

In [18]:
res = minimize(
    objective_slsqp, 
    init_weights, 
    method='SLSQP', 
    bounds=bounds, 
    constraints=cons,
    options={'maxiter': 500}
)

In [19]:
# 4. Process and Normalize the Final Best Weights
best_weights = res.x

In [20]:
# 5. Print Results
print("Optimization Successful:", res.success)
print(f"Optimized Blending Weights: {np.round(best_weights, 4)}")

Optimization Successful: True
Optimized Blending Weights: [0.0179 0.0228 0.0013 0.0077 0.032  0.0083 0.0097 0.0345 0.0022 0.0189
 0.0584 0.0115 0.0384 0.0568 0.017  0.0394 0.     0.0091 0.0281 0.0414
 0.0135 0.0049 0.0526 0.0244 0.0636 0.0255 0.0268 0.0489 0.0255 0.0286
 0.0255 0.0268 0.033  0.0389 0.0204 0.0379 0.0475]


In [21]:
# Calculate the actual best ROC-AUC score using the optimized weights
X_blend = X.values @ best_weights
best_auc = roc_auc_score(y.values, X_blend)

print(f"Best ROC-AUC Score: {best_auc:.4f}")

Best ROC-AUC Score: 0.9459


In [22]:
# Compute the final blended predictions
final_blend = X_test.values @ best_weights

In [23]:
submit["is_fraud"] = final_blend

In [25]:
submit.to_csv(competition_path/"Blending SLSQP v3.csv",index=False)

In [26]:
res.message

'Optimization terminated successfully'